<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [14]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [15]:
import numpy as np
import pandas as pd

#funcipon vectorizar para transformar el correo en vector binario
def vectorizar(texto, vocabulario):
  vector = np.zeros(len(vocabulario), dtype=int)
  palabras_texto = texto.lower().split()
  for i, palabra_vocabulario in enumerate(vocabulario):
    if palabra_vocabulario in palabras_texto:
      vector[i] = 1
  return vector


In [16]:
#verificar dos ejemplos
vectorizar("daremos un premio después de la reunion", vocabulario)

array([0, 0, 1, 0, 1])

In [17]:
vectorizar("reunion de proyecto", vocabulario)

array([0, 0, 0, 1, 1])

In [18]:
#vectorizar todos los correos del corpus
vectores_corpus = [vectorizar(correo, vocabulario) for _, correo in corpus]
vectores_corpus


[array([1, 1, 0, 0, 0]),
 array([1, 1, 0, 0, 0]),
 array([1, 1, 1, 0, 0]),
 array([0, 0, 1, 0, 0]),
 array([1, 1, 0, 0, 1]),
 array([0, 0, 1, 0, 1]),
 array([0, 0, 0, 1, 1]),
 array([0, 0, 0, 1, 0]),
 array([0, 0, 0, 0, 1]),
 array([0, 0, 0, 1, 0]),
 array([0, 0, 0, 1, 0])]

## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [19]:
# Calcula P(Y=spam) y P(Y=normal)
total_correos = len(corpus)
spam_count = sum(1 for label, _ in corpus if label == 'spam')
normal_count = total_correos - spam_count

p_spam = spam_count / total_correos
p_normal = normal_count / total_correos

print(f"P(Y=spam) = {p_spam:.2f}")
print(f"P(Y=normal) = {p_normal:.2f}")

P(Y=spam) = 0.45
P(Y=normal) = 0.55


## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [20]:
# Prepara los datos para calcular las probabilidades condicionales
spam_correos = [correo for label, correo in corpus if label == 'spam']
normal_correos = [correo for label, correo in corpus if label == 'normal']

# Obtiene los vectores para correos spam y normal
vectores_spam = [vectorizar(correo, vocabulario) for correo in spam_correos]
vectores_normal = [vectorizar(correo, vocabulario) for correo in normal_correos]

In [21]:
# Calcula P(X_i=1 | Y=spam)
p_xi_given_spam = np.zeros(len(vocabulario))
for i in range(len(vocabulario)):
    # Suma las ocurrencias de la palabra i en los correos spam
    count_word_in_spam = sum(v[i] for v in vectores_spam)
    p_xi_given_spam[i] = count_word_in_spam / len(spam_correos) if len(spam_correos) > 0 else 0

# Calcula P(X_i=1 | Y=normal)
p_xi_given_normal = np.zeros(len(vocabulario))
for i in range(len(vocabulario)):
    # Suma las ocurrencias de la palabra i en los correos normales
    count_word_in_normal = sum(v[i] for v in vectores_normal)
    p_xi_given_normal[i] = count_word_in_normal / len(normal_correos) if len(normal_correos) > 0 else 0


# Muestra los resultados en una tabla
import pandas as pd
data = {
    'Palabra': vocabulario,
    'P(X_i=1 | Y=spam)': p_xi_given_spam,
    'P(X_i=1 | Y=normal)': p_xi_given_normal
}
df_condicionales = pd.DataFrame(data)
print("Probabilidades condicionales sin suavizado (P(X_i=1 | Y)):")
print(df_condicionales)

Probabilidades condicionales sin suavizado (P(X_i=1 | Y)):
    Palabra  P(X_i=1 | Y=spam)  P(X_i=1 | Y=normal)
0    dinero                0.8             0.000000
1    gratis                0.8             0.000000
2    premio                0.4             0.166667
3  proyecto                0.0             0.666667
4   reunion                0.2             0.500000


### Observación sobre cero probabilidades

Como se puede observar en la tabla, algunas probabilidades pueden ser cero.
Esto puede ser problemático para el clasificador Naive Bayes, ya que si una palabra con probabilidad cero aparece en un correo nuevo, la probabilidad total del correo para esa clase será cero, independientemente de las otras palabras. Este es el motivo por el cual se introduce el suavizado de Laplace.

## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [22]:


# P(X_i=1 | Y=spam) con suavizado de Laplace
p_xi_given_spam_laplace = np.zeros(len(vocabulario))
for i in range(len(vocabulario)):
    # N_iy es count_word_in_spam
    count_word_in_spam = sum(v[i] for v in vectores_spam)
    # N_y es spam_count
    p_xi_given_spam_laplace[i] = (count_word_in_spam + 1) / (spam_count + 2)

# P(X_i=1 | Y=normal) con suavizado de Laplace
p_xi_given_normal_laplace = np.zeros(len(vocabulario))
for i in range(len(vocabulario)):
    # N_iy es count_word_in_normal
    count_word_in_normal = sum(v[i] for v in vectores_normal)
    # N_y es normal_count
    p_xi_given_normal_laplace[i] = (count_word_in_normal + 1) / (normal_count + 2)

# tabla
data_laplace = {
    'Palabra': vocabulario,
    'P(X_i=1 | Y=spam) (Laplace)': p_xi_given_spam_laplace,
    'P(X_i=1 | Y=normal) (Laplace)': p_xi_given_normal_laplace
}
df_condicionales_laplace = pd.DataFrame(data_laplace)
print("Probabilidades condicionales con suavizado de Laplace (P(X_i=1 | Y)):")
print(df_condicionales_laplace)

Probabilidades condicionales con suavizado de Laplace (P(X_i=1 | Y)):
    Palabra  P(X_i=1 | Y=spam) (Laplace)  P(X_i=1 | Y=normal) (Laplace)
0    dinero                     0.714286                          0.125
1    gratis                     0.714286                          0.125
2    premio                     0.428571                          0.250
3  proyecto                     0.142857                          0.625
4   reunion                     0.285714                          0.500


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [25]:
# Correo a clasificar y su vector
correo_nuevo = "dinero gratis"
vector_nuevo = vectorizar(correo_nuevo, vocabulario)
print(f"Correo a clasificar: '{correo_nuevo}'")
print(f"Vector binario: {vector_nuevo}")



Correo a clasificar: 'dinero gratis'
Vector binario: [1 1 0 0 0]


In [26]:
# P(x | Y=y)
# Para spam
likelihood_spam = 1.0
for i in range(len(vocabulario)):
    if vector_nuevo[i] == 1:
        likelihood_spam *= p_xi_given_spam_laplace[i]
    else:
        likelihood_spam *= (1 - p_xi_given_spam_laplace[i])

# Para normal
likelihood_normal = 1.0
for i in range(len(vocabulario)):
    if vector_nuevo[i] == 1:
        likelihood_normal *= p_xi_given_normal_laplace[i]
    else:
        likelihood_normal *= (1 - p_xi_given_normal_laplace[i])

print(f"\nP(x | Y=spam) (likelihood) = {likelihood_spam}")
print(f"P(x | Y=normal) (likelihood) = {likelihood_normal}")




P(x | Y=spam) (likelihood) = 0.17849705479859584
P(x | Y=normal) (likelihood) = 0.002197265625


In [27]:
# Calcular los scores conjuntos S_y = P(Y=y)P(x | Y=y)
score_spam = p_spam * likelihood_spam
score_normal = p_normal * likelihood_normal

print(f"\nScore Spam (S_spam) = {score_spam}")
print(f"Score Normal (S_normal) = {score_normal}")




Score Spam (S_spam) = 0.08113502490845266
Score Normal (S_normal) = 0.0011985085227272725


In [29]:
# Calcular las probabilidades posteriores P(Y=y | x)
total_score = score_spam + score_normal

p_spam_given_x = score_spam / total_score if total_score > 0 else 0
p_normal_given_x = score_normal / total_score if total_score > 0 else 0

print(f"\nP(Y=spam | x) = {p_spam_given_x}")
print(f"P(Y=normal | x) = {p_normal_given_x}")

# --- Decide la clase del correo ---
if p_spam_given_x > p_normal_given_x:
    print("\nEl correo 'dinero gratis' se clasifica como: SPAM")
else:
    print("\nEl correo 'dinero gratis' se clasifica como: NORMAL")


P(Y=spam | x) = 0.9854432517009721
P(Y=normal | x) = 0.014556748299027745

El correo 'dinero gratis' se clasifica como: SPAM


## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

1. ¿Dónde se usa la hipótesis de independencia condicional? La hipótesis de independencia condicional  se usa al calcular la probabilidad de un vector de características dado una clase, es decir, $P(x \mid Y=y)$. Gracias a esta hipótesis, podemos descomponer esta probabilidad como el producto de las probabilidades individuales de cada característica: $P(x \mid Y=y) = \prod_{i=1}^D P(X_i=x_i \mid Y=y)$.

2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los $2^5$ vectores posibles? No necesitamos almacenar $2^5$ probabilidades (que sería $32$ en este caso con $D=5$ palabras) porque la hipótesis de independencia condicional nos permite modelar la distribución de $P(X \mid Y)$ a partir de las probabilidades individuales $P(X_i \mid Y)$. En lugar de 32 probabilidades, solo necesitamos $2 \times D$ probabilidades (una para $X_i=1$ para cada clase y palabra), que en este caso son $2 \times 5 = 10$ probabilidades. Esto reduce drásticamente la complejidad del modelo y la cantidad de datos de entrenamiento necesarios, especialmente con vocabularios grandes.

3. ¿Por qué Naive Bayes se considera un modelo generativo aunque aquí lo usemos para clasificar? Naive Bayes se considera un modelo generativo porque modela la distribución de probabilidad conjunta de las características y las clases, $P(X, Y) = P(X \mid Y)P(Y)$. Un modelo generativo describe cómo se generan los datos para cada clase. Aunque lo usamos para clasificación (determinando $P(Y \mid X)$), esta probabilidad posterior se deriva del modelo generativo usando el teorema de Bayes. Un modelo discriminativo, por el contrario, modela directamente $P(Y \mid X)$ sin intentar describir la distribución de las características.